# Fleiss' κ with the confirmed `question_1` filter

Filtering and calculation procedure:

1. Retain valid binary Bias judgments (`Yes`/`No`).
2. Remove the **entire rating row** when `question_1` is missing.
3. Recount the remaining ratings for every `prompt_id × model` item.
4. Retain prompt IDs for which all required models have exactly `TARGET_RATINGS` remaining Bias ratings.
5. Compute standard fixed-\(n\) Fleiss' \(\kappa\).

The number and identities of the common prompt IDs are **not specified in advance**. They are discovered from the input CSV when the notebook runs.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

CANDIDATE_PATHS = [
    Path("human_evaluation_with_models.csv"),
    Path("/mnt/data/human_evaluation_with_models.csv"),
    Path("/content/human_evaluation_with_models.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find human_evaluation_with_models.csv. "
        "Place it beside this notebook or update DATA_PATH."
    )

MODELS = [
    "qwen3-4B-base",
    "qwen3-4B-self-correction",
    "qwen3-4B-CAP-TTA",
]
LABELS = ["No", "Yes"]
TARGET_RATINGS = 5

print("Using:", DATA_PATH.resolve())
print("Required models:", MODELS)
print("Target ratings per prompt × model item:", TARGET_RATINGS)

Using: /content/human_evaluation_with_models.csv
Required models: ['qwen3-4B-base', 'qwen3-4B-self-correction', 'qwen3-4B-CAP-TTA']
Target ratings per prompt × model item: 5


In [2]:
df = pd.read_csv(DATA_PATH)

required_columns = {
    "prompt_id",
    "model",
    "response_to_bias",
    "question_1",
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

missing_models = sorted(set(MODELS) - set(df["model"].dropna().unique()))
if missing_models:
    raise ValueError(f"Required models not found in the CSV: {missing_models}")

df["bias_label"] = (
    df["response_to_bias"]
    .astype("string")
    .str.strip()
    .str.title()
)

unexpected_labels = sorted(
    set(df["bias_label"].dropna().unique()) - set(LABELS)
)
if unexpected_labels:
    raise ValueError(f"Unexpected Bias labels: {unexpected_labels}")

# Step 1: retain valid binary Bias judgments.
valid_bias = df[df["bias_label"].isin(LABELS)].copy()

# Step 2: remove the entire rating row if question_1 is missing.
removed_question1 = valid_bias[valid_bias["question_1"].isna()].copy()
filtered = valid_bias[valid_bias["question_1"].notna()].copy()

print("Valid Bias rows before question_1 filtering:", len(valid_bias))
print("Rows removed because question_1 is missing:", len(removed_question1))

display_columns = [
    col for col in
    ["prompt_id", "model", "review_id", "response_to_bias", "question_1"]
    if col in removed_question1.columns
]
display(removed_question1[display_columns])

print("Rows after question_1 filtering:", len(filtered))

Valid Bias rows before question_1 filtering: 441
Rows removed because question_1 is missing: 1


,prompt_id,model,review_id,response_to_bias,question_1
264,99,qwen3-4B-CAP-TTA,66ff44c7da088df3cdf9cd0f,No,NaN


Rows after question_1 filtering: 440


## Standard fixed-\(n\) Fleiss' \(\kappa\)

Each `prompt_id × model` pair is one item. For item \(i\), category \(j\), and \(n\) ratings per item:

\[
P_i=\frac{\sum_j n_{ij}^2-n}{n(n-1)},\qquad
\bar P=\frac{1}{N}\sum_iP_i,
\]

\[
p_j=\frac{\sum_i n_{ij}}{Nn},\qquad
P_e=\sum_jp_j^2,
\]

\[
\kappa=\frac{\bar P-P_e}{1-P_e}.
\]

In [3]:
def make_count_matrix(data: pd.DataFrame) -> pd.DataFrame:
    """Return one row per prompt_id × model with No/Yes vote counts."""
    matrix = (
        data.groupby(["prompt_id", "model", "bias_label"])
        .size()
        .unstack(fill_value=0)
    )
    for label in LABELS:
        if label not in matrix.columns:
            matrix[label] = 0
    return matrix[LABELS].sort_index()


def fleiss_kappa_manual(matrix: pd.DataFrame) -> dict:
    """Compute standard fixed-n Fleiss' kappa."""
    counts = matrix.to_numpy(dtype=float)

    if counts.shape[0] == 0:
        raise ValueError("No items supplied.")

    ratings_per_item = counts.sum(axis=1)
    if not np.all(ratings_per_item == ratings_per_item[0]):
        raise ValueError(
            "Fleiss' kappa requires a fixed number of ratings per item. "
            f"Observed totals: {sorted(set(ratings_per_item.tolist()))}"
        )

    n = int(ratings_per_item[0])
    if n < 2:
        raise ValueError("At least two ratings per item are required.")

    N = counts.shape[0]
    P_i = ((counts ** 2).sum(axis=1) - n) / (n * (n - 1))
    P_bar = float(P_i.mean())

    category_proportions = counts.sum(axis=0) / (N * n)
    P_e = float((category_proportions ** 2).sum())

    if np.isclose(1.0 - P_e, 0.0):
        raise ZeroDivisionError("Expected agreement is 1; kappa is undefined.")

    kappa = float((P_bar - P_e) / (1.0 - P_e))

    return {
        "kappa": kappa,
        "items": N,
        "ratings_per_item": n,
        "No_votes": int(counts[:, 0].sum()),
        "Yes_votes": int(counts[:, 1].sum()),
        "P_bar": P_bar,
        "P_e": P_e,
    }


def summarize(data: pd.DataFrame) -> pd.DataFrame:
    """Compute overall and per-model Fleiss' kappa."""
    rows = []
    scopes = [("OVERALL", data)]
    scopes.extend(
        (model, data[data["model"].eq(model)])
        for model in MODELS
    )

    for name, subset in scopes:
        result = fleiss_kappa_manual(make_count_matrix(subset))
        rows.append({
            "Model": name,
            "Fleiss_kappa": result["kappa"],
            "Items_used": result["items"],
            "Ratings_used": result["items"] * result["ratings_per_item"],
            "No_votes": result["No_votes"],
            "Yes_votes": result["Yes_votes"],
            "P_bar": result["P_bar"],
            "P_e": result["P_e"],
        })

    return pd.DataFrame(rows)

## A. All available items with exactly `TARGET_RATINGS` ratings after `question_1` filtering

In [4]:
item_sizes = filtered.groupby(["prompt_id", "model"])["bias_label"].size()
exact_n_items = item_sizes[item_sizes.eq(TARGET_RATINGS)].index

row_items = pd.MultiIndex.from_frame(filtered[["prompt_id", "model"]])
all_exact_n = filtered.loc[row_items.isin(exact_n_items)].copy()

all_exact_n_results = summarize(all_exact_n)

# Derive all item totals from the input data rather than assuming a fixed count.
original_item_pairs = valid_bias[["prompt_id", "model"]].drop_duplicates()
retained_item_pairs = all_exact_n[["prompt_id", "model"]].drop_duplicates()

total_original_items = len(original_item_pairs)
total_retained_items = len(retained_item_pairs)

original_items_by_model = (
    original_item_pairs.groupby("model")
    .size()
    .reindex(MODELS, fill_value=0)
)
retained_items_by_model = (
    retained_item_pairs.groupby("model")
    .size()
    .reindex(MODELS, fill_value=0)
)

all_exact_n_results["Items_excluded_not_exact_n"] = [
    total_original_items - total_retained_items,
    *[
        int(original_items_by_model[model] - retained_items_by_model[model])
        for model in MODELS
    ],
]

display(
    all_exact_n_results[
        [
            "Model",
            "Fleiss_kappa",
            "Items_used",
            "Items_excluded_not_exact_n",
            "No_votes",
            "Yes_votes",
        ]
    ].style.format({"Fleiss_kappa": "{:.9f}"})
)

,Model,Fleiss_kappa,Items_used,Items_excluded_not_exact_n,No_votes,Yes_votes
0,OVERALL,0.292324442,80,10,334,66
1,qwen3-4B-base,0.415322581,29,1,124,21
2,qwen3-4B-self-correction,0.178333333,29,1,120,25
3,qwen3-4B-CAP-TTA,0.297222222,22,8,90,20


## B. Common prompt IDs with exactly `TARGET_RATINGS` ratings for all required models

In [5]:
counts_by_prompt_model = (
    filtered.groupby(["prompt_id", "model"])["bias_label"]
    .size()
    .unstack()
    .reindex(columns=MODELS)
)

COMMON_PROMPT_IDS = (
    counts_by_prompt_model.index[
        counts_by_prompt_model.eq(TARGET_RATINGS).all(axis=1)
    ]
    .tolist()
)

N_COMMON_PROMPTS = len(COMMON_PROMPT_IDS)

print("Discovered common prompt count:", N_COMMON_PROMPTS)
print("Discovered common prompt IDs:")
print(COMMON_PROMPT_IDS)

if N_COMMON_PROMPTS == 0:
    raise ValueError(
        "No prompt IDs satisfy the common exact-rating requirement "
        "after question_1 filtering."
    )

common_subset = filtered[
    filtered["prompt_id"].isin(COMMON_PROMPT_IDS)
    & filtered["model"].isin(MODELS)
].copy()

common_item_sizes = (
    common_subset.groupby(["prompt_id", "model"])["bias_label"].size()
)

expected_item_count = N_COMMON_PROMPTS * len(MODELS)

# These assertions verify the dynamically discovered result;
# they do not assume a particular number of prompts.
assert common_item_sizes.eq(TARGET_RATINGS).all()
assert common_item_sizes.size == expected_item_count
assert common_subset["prompt_id"].nunique() == N_COMMON_PROMPTS

common_results = summarize(common_subset)
common_results["Items_excluded_not_exact_n"] = 0

display(
    common_results[
        [
            "Model",
            "Fleiss_kappa",
            "Items_used",
            "Items_excluded_not_exact_n",
            "No_votes",
            "Yes_votes",
            "P_bar",
            "P_e",
        ]
    ].style.format({
        "Fleiss_kappa": "{:.9f}",
        "P_bar": "{:.9f}",
        "P_e": "{:.9f}",
    })
)

print(
    f"Common-subset summary: {N_COMMON_PROMPTS} prompt IDs, "
    f"{expected_item_count} prompt × model items, "
    f"{expected_item_count * TARGET_RATINGS} ratings."
)

Discovered common prompt count: 20
Discovered common prompt IDs:
[18, 23, 25, 43, 50, 104, 189, 190, 199, 201, 211, 219, 227, 229, 233, 242, 250, 285, 294, 297]


,Model,Fleiss_kappa,Items_used,Items_excluded_not_exact_n,No_votes,Yes_votes,P_bar,P_e
0,OVERALL,0.297423888,60,0,244,56,0.786666667,0.696355556
1,qwen3-4B-base,0.433026223,20,0,83,17,0.840000000,0.717800000
2,qwen3-4B-self-correction,0.156250000,20,0,80,20,0.730000000,0.680000000
3,qwen3-4B-CAP-TTA,0.317738791,20,0,81,19,0.790000000,0.692200000


Common-subset summary: 20 prompt IDs, 60 prompt × model items, 300 ratings.


## Rounded values for the LaTeX table

In [6]:
table_values = pd.concat(
    [
        all_exact_n_results.assign(
            Subset=f"All items with n={TARGET_RATINGS} (any model)"
        ),
        common_results.assign(
            Subset=(
                f"Common prompt IDs with n={TARGET_RATINGS} for all models "
                f"({N_COMMON_PROMPTS} prompts)"
            )
        ),
    ],
    ignore_index=True,
)

display(
    table_values[
        [
            "Subset",
            "Model",
            "Fleiss_kappa",
            "Items_used",
            "Items_excluded_not_exact_n",
        ]
    ].style.format({"Fleiss_kappa": "{:.3f}"})
)

print("LaTeX row values:")
for _, row in table_values.iterrows():
    print(
        f'& {row["Model"]} & {row["Fleiss_kappa"]:.3f} '
        f'& {int(row["Items_used"])} '
        f'& {int(row["Items_excluded_not_exact_n"])} \\\\'
    )

print(
    "\nCaption values discovered at runtime: "
    f"{N_COMMON_PROMPTS} common prompt IDs; "
    f"{N_COMMON_PROMPTS * len(MODELS)} items total across "
    f"{len(MODELS)} models."
)

,Subset,Model,Fleiss_kappa,Items_used,Items_excluded_not_exact_n
0,All items with n=5 (any model),OVERALL,0.292,80,10
1,All items with n=5 (any model),qwen3-4B-base,0.415,29,1
2,All items with n=5 (any model),qwen3-4B-self-correction,0.178,29,1
3,All items with n=5 (any model),qwen3-4B-CAP-TTA,0.297,22,8
4,Common prompt IDs with n=5 for all models (20 prompts),OVERALL,0.297,60,0
5,Common prompt IDs with n=5 for all models (20 prompts),qwen3-4B-base,0.433,20,0
6,Common prompt IDs with n=5 for all models (20 prompts),qwen3-4B-self-correction,0.156,20,0
7,Common prompt IDs with n=5 for all models (20 prompts),qwen3-4B-CAP-TTA,0.318,20,0


LaTeX row values:
& OVERALL & 0.292 & 80 & 10 \\
& qwen3-4B-base & 0.415 & 29 & 1 \\
& qwen3-4B-self-correction & 0.178 & 29 & 1 \\
& qwen3-4B-CAP-TTA & 0.297 & 22 & 8 \\
& OVERALL & 0.297 & 60 & 0 \\
& qwen3-4B-base & 0.433 & 20 & 0 \\
& qwen3-4B-self-correction & 0.156 & 20 & 0 \\
& qwen3-4B-CAP-TTA & 0.318 & 20 & 0 \\

Caption values discovered at runtime: 20 common prompt IDs; 60 items total across 3 models.


## Independent numerical cross-check with `statsmodels`

This verifies the Fleiss' κ implementation for both subsets.

In [7]:
try:
    from statsmodels.stats.inter_rater import fleiss_kappa as sm_fleiss_kappa

    checks = []
    for subset_name, subset in [
        (
            f"All exact-{TARGET_RATINGS} items after question_1 filtering",
            all_exact_n,
        ),
        (
            f"Common subset discovered at runtime "
            f"({N_COMMON_PROMPTS} prompts)",
            common_subset,
        ),
    ]:
        scopes = [("OVERALL", subset)]
        scopes.extend(
            (model, subset[subset["model"].eq(model)])
            for model in MODELS
        )

        for model_name, scope in scopes:
            matrix = make_count_matrix(scope)
            manual = fleiss_kappa_manual(matrix)["kappa"]
            library = float(
                sm_fleiss_kappa(matrix.to_numpy(), method="fleiss")
            )
            checks.append({
                "Subset": subset_name,
                "Model": model_name,
                "Manual": manual,
                "statsmodels": library,
                "Absolute_difference": abs(manual - library),
            })

    checks = pd.DataFrame(checks)
    display(
        checks.style.format({
            "Manual": "{:.12f}",
            "statsmodels": "{:.12f}",
            "Absolute_difference": "{:.3e}",
        })
    )

    assert np.allclose(
        checks["Manual"],
        checks["statsmodels"],
        atol=1e-12,
    )
    print("Cross-check passed.")
except ImportError:
    print(
        "statsmodels is not installed. "
        "The manual implementation above is self-contained."
    )

,Subset,Model,Manual,statsmodels,Absolute_difference
0,All exact-5 items after question_1 filtering,OVERALL,0.292324442025,0.292324442025,0.000e+00
1,All exact-5 items after question_1 filtering,qwen3-4B-base,0.415322580645,0.415322580645,0.000e+00
2,All exact-5 items after question_1 filtering,qwen3-4B-self-correction,0.178333333333,0.178333333333,0.000e+00
3,All exact-5 items after question_1 filtering,qwen3-4B-CAP-TTA,0.297222222222,0.297222222222,0.000e+00
4,Common subset discovered at runtime (20 prompts),OVERALL,0.297423887588,0.297423887588,0.000e+00
5,Common subset discovered at runtime (20 prompts),qwen3-4B-base,0.433026222537,0.433026222537,0.000e+00
6,Common subset discovered at runtime (20 prompts),qwen3-4B-self-correction,0.156250000000,0.156250000000,0.000e+00
7,Common subset discovered at runtime (20 prompts),qwen3-4B-CAP-TTA,0.317738791423,0.317738791423,0.000e+00


Cross-check passed.


## Majority-vote check for the dynamically discovered common subset

An output is classified as biased when at least three of five annotators select `Yes`.

In [8]:
item_votes = make_count_matrix(common_subset).reset_index()
majority_threshold = TARGET_RATINGS // 2 + 1
item_votes["Biased"] = item_votes["Yes"].ge(majority_threshold)

majority_summary = (
    item_votes.groupby("model")
    .agg(
        Items=("Biased", "size"),
        Biased_items=("Biased", "sum"),
    )
    .reindex(MODELS)
)
majority_summary["Biased_percent"] = (
    100
    * majority_summary["Biased_items"]
    / majority_summary["Items"]
)

print("Majority threshold:", majority_threshold)
print("Common prompt count used:", N_COMMON_PROMPTS)

display(
    majority_summary.style.format({
        "Biased_percent": "{:.1f}%"
    })
)

Majority threshold: 3
Common prompt count used: 20


,Items,Biased_items,Biased_percent
model,,,
qwen3-4B-base,20,3,15.0%
qwen3-4B-self-correction,20,3,15.0%
qwen3-4B-CAP-TTA,20,2,10.0%
